In [21]:
#Importando Bibliotecas que estou utilizando para a análise
import pandas as pd
from ydata_profiling import ProfileReport
import os

In [22]:
#Caminho da Base de dados
caminho_dados = r"C:\Users\gabri\OneDrive\Belgeler\GitHub\Python\AnalisesRedebras\Referência\Dados\RelPedidoCli.xls"
df_comercial = pd.read_excel(caminho_dados)

# Sobrescreve o dataframe removendo a última linha (Linha total que o próprio ERP ja envia junto)
df_comercial = df_comercial.iloc[:-1]

In [23]:
## ALTERAÇÕES DE LIMPEZA E FORMATAÇÃO ##

#Modificando Número para Texto (Pedidos e N.F -> Precisam ser identificados como texto para a análise)
colunas_texto = ['PEDIDO', 'NF']
for col in colunas_texto:
    df_comercial[col] = (
        df_comercial[col].astype(str)
    )

#------------------------------------------------------------------------------#
# Forçamos as colunas a serem strings para limpeza manual 
# Formatando os Valores de Preços e Custos
colunas_preco = ['TOTAL CUSTO', 'TOTAL ITENS', 'TOTAL PEDIDO', 'QTDE VENDA']
for col in colunas_preco:
    df_comercial[col] = (
        df_comercial[col].astype(str)
    )

for col in colunas_preco:
    # 1. Troca a vírgula pelo ponto decimal (1200,50 -> 1200.50)
    df_comercial[col] = df_comercial[col].str.replace(',', '.', regex=False)
    # 2. Converte para número real (float)
    df_comercial[col] = pd.to_numeric(df_comercial[col], errors='coerce')
    # 3. Arredonda para 2 casas decimais
    df_comercial[colunas_preco] = df_comercial[colunas_preco].round(2)

#------------------------------------------------------------------------------#
#Formatando Colunas de Datas
# Modificando as colunas de Data para o tipo Data
colunas_data = ['DATA', 'NF. EMISSAO']

for col in colunas_data:
    df_comercial[col] = (
    pd.to_datetime(df_comercial[col], dayfirst=True, errors='coerce')
    )

#------------------------------------------------------------------------------#
#Limpeza de Colunas não necessárias para o projeto (Deletando Colunas)
colunas_desnecessarias = ['EMP', 'REPRESENTANTE', 'TIPO', 'INTEGRAÇÃO', 'OUTRAS DESP', 'VLR. FRETE', 'DT. RETIRADA', 'FRETE TYPE', 'FIN',
                           'TOTAL ITENS C/ IPI', 'TOTAL ITENS', 'MKP']

df_comercial = df_comercial.drop(columns=colunas_desnecessarias, errors='ignore')   

#------------------------------------------------------------------------------#
# Criação de colunas
# Lucro R$ (Para saber o lucro de cada venda em dinheiro)
df_comercial['LUCRO R$'] = (
    df_comercial['TOTAL PEDIDO'] - df_comercial['TOTAL CUSTO']
)
df_comercial['LUCRO R$'] = df_comercial['LUCRO R$'].round(2)

# Total de Vendas Mês (Para saber o total de vendas do mês, somando o valor de cada venda)
df_comercial['TOTAL VENDAS MÊS'] = df_comercial['TOTAL PEDIDO'].sum()

# Formatando a coluna de Data para o formato brasileiro (dd/mm/yyyy)
df_comercial['DATA'] = df_comercial['DATA'].dt.strftime('%d/%m/%Y')

# Formatando a coluna de Data NF. EMISSAO para o formato brasileiro (dd/mm/yyyy)
df_comercial['NF. EMISSAO'] = df_comercial['NF. EMISSAO'].dt.strftime('%d/%m/%Y')

##--------------------------------------##
# Calculando o MKP REAL de cada venda
# 1.Markup com o multiplicador real (Preço de Venda / Custo) para comparar com o MKP do ERP
df_comercial['MKP REAL'] = df_comercial['TOTAL PEDIDO'] / df_comercial['TOTAL CUSTO']

# 2.Calculando o MKP REAL em % (Multiplicador real - 1) * 100 para comparar com o MKP do ERP
df_comercial['MKP REAL %'] = (df_comercial['MKP REAL'] - 1) * 100

# 3.Tratamento de dados em caso de erro
df_comercial['MKP REAL %'] = df_comercial['MKP REAL %'].replace([float('inf'), float('-inf')], 0)  # Substitui inf e -inf por 0

# Preenchendo os valores nulos da coluna MKP com 0 (zero) e colocando o formato de porcentagem com 2 casas decimais
df_comercial['MKP REAL %'] = df_comercial['MKP REAL %'].fillna(0)
df_comercial['MKP REAL %'] = df_comercial['MKP REAL %'].round(2)

#Alterando nome da coluna Clientes
df_comercial = df_comercial.rename(columns={'NOME': 'CLIENTES'})



In [24]:
# Criação de DataFrame Complementar para análise comercial por cidades do Brasil

df_analise_estado = df_comercial.groupby('UF').agg({
    'PEDIDO': 'count',
    'TOTAL PEDIDO': 'sum',
    'LUCRO R$': 'sum',
    'MKP REAL %': 'mean'
}).reset_index()

# Renomeando as colunas do DataFrame de análise por cidades para melhor entendimento

df_analise_estado = df_analise_estado.rename(columns={
    'PEDIDO': 'QUANTIDADE DE PEDIDOS',
    'TOTAL PEDIDO': 'TOTAL FATURAMENTO',
    'LUCRO R$': 'LUCRO TOTAL',
    'MKP REAL %': 'MKP MÉDIO DE VENDA %'
})

#Calculos
# Calculando o Ticket Médio por cidade (Total Faturamento / Quantidade de Pedidos)

df_analise_estado['TICKET MÉDIO'] = df_analise_estado['TOTAL FATURAMENTO'] / df_analise_estado['QUANTIDADE DE PEDIDOS']

# Tratando os dados para evitar valores nulos e arredondando as colunas para 2 casas decimais
df_analise_estado['MKP MÉDIO DE VENDA %'] = df_analise_estado['MKP MÉDIO DE VENDA %'].fillna(0)
df_analise_estado['MKP MÉDIO DE VENDA %'] = df_analise_estado['MKP MÉDIO DE VENDA %'].round(2)
df_analise_estado['TOTAL FATURAMENTO'] = df_analise_estado['TOTAL FATURAMENTO'].round(2)
df_analise_estado['LUCRO TOTAL'] = df_analise_estado['LUCRO TOTAL'].round(2)
df_analise_estado['TICKET MÉDIO'] = df_analise_estado['TICKET MÉDIO'].round(2)

# Invesigando o Vendedor que mais atende determinado Estado (UF)

#✅ Passo 1 — contar vendas por cidade + vendedor
df_vendedor = df_comercial.groupby(['UF', 'VENDEDOR']).size().reset_index(name='QUANTIDADE DE PEDIDOS')

#✅ Passo 2 — pegar o TOP vendedor por cidade
df_top_vendedor = df_vendedor.sort_values('QUANTIDADE DE PEDIDOS', ascending=False)\
    .drop_duplicates('UF')

#✅ Passo 3 — juntar com df_analise_cidades para ter o nome do vendedor junto com as análises por cidade
df_analise_estado = df_analise_estado.merge(
    df_top_vendedor[['UF', 'VENDEDOR']],
    on='UF',
    how='left'
)



In [25]:
df_analise_estado

,UF,QUANTIDADE DE PEDIDOS,TOTAL FATURAMENTO,LUCRO TOTAL,MKP MÉDIO DE VENDA %,TICKET MÉDIO,VENDEDOR
0,AL,3,5419.87,1978.41,60.68,1806.62,EUDES
1,AM,5,43829.44,8928.19,46.87,8765.89,EUDES
2,BA,34,94752.05,36907.00,73.64,2786.82,ANDERSON
3,CE,3,24564.64,10963.25,65.37,8188.21,MATHEUS
4,DF,6,37153.68,7459.71,64.03,6192.28,EUDES
5,ES,20,94331.72,36082.75,70.71,4716.59,ANDERSON
6,GO,10,101726.02,-283964.27,38.91,10172.60,MATHEUS
7,MA,1,680.55,307.59,82.47,680.55,ECOMMERCE
8,MG,58,254229.20,83676.39,226.14,4383.26,GERSON
9,MS,4,32894.21,10124.07,46.84,8223.55,ANDERSON
